# Funny Python — пробный ноутбук

Ядро: **Funny Python (3.15)** (`funnypy`) — питон-форк с многострочными
лямбдами и паттерн-деструктуризацией.

- Ноутбук уже привязан к ядру `funnypy` (метаданные `kernelspec`), но если
  открыть новый — выбрать `Kernel → Change Kernel → Funny Python (3.15)`.
- Ядро держит собранный бинарник: **после пересборки `cpython-main`**
  (`make regen-pegen && make -j$(nproc)`) сделать `Kernel → Restart`, иначе
  правки грамматики не подхватятся.
- Установка окружения: venv `~/.local/share/funnypy/venv`, ядро
  `~/.local/share/jupyter/kernels/funnypy`.

In [ ]:
import sys
print(sys.version)
print(sys.executable)

## 1. Многострочная лямбда: `def(аргументы): <блок>`

Полноценный блок: циклы, `try`, `yield` (делает генератором), замыкания.

In [ ]:
fib = def(n):
    a, b = 0, 1
    for _ in range(n):
        yield a
        a, b = b, a + b

list(fib(10))

In [ ]:
# return-позиция и замыкание
def make_mul(k):
    return def(x):
        return x * k

triple = make_mul(3)
print(triple.__name__)   # <lambda> в return-позиции
[triple(i) for i in range(5)]

## 2. Паттерн-деструктуризация: `match PATTERN = value`

Слева — любой паттерн из `match..case`; при несовпадении — `MatchError`.

In [ ]:
payload = {"user": {"name": "ada", "id": 7}, "tags": [1, 2, 3]}

match {"user": {"name": name}, "tags": [first, *rest]} = payload
print(name, first, rest)

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

match Point(x=px, y=py) = Point(1, 2)
print(px, py)

In [1]:
# or-паттерн: все альтернативы связывают одно имя
match {"a": a} | {"b": a} = {"b": 9}
a

9

In [1]:
try:
    match {"x": x} = {"y": 1}
except MatchError as exc:
    print("не совпало ->", type(exc).__name__, "|", isinstance(exc, Exception))

не совпало -> MatchError | True


In [2]:
# irrefutable-паттерн не должен падать; однострочное тело тоже работает
one_liner = def(x): return x + 1
match _ = "всегда совпадает"
one_liner(41)

42

## 3. Оператор `..`: пайплайн, placeholder, pipe-match

`..` — только с новой строки; в терминальном REPL не работает (только файлы и ячейки).

In [ ]:
square = def(x): return x ** 2
is_even = def(x): return x % 2 == 0

[1, 2, 3, 4, 5]
..map(square)
..filter(is_even)
..list

In [ ]:
for_each = def(f, data):
    for x in data:
        f(x)

[1, 2, 3]
..for_each(..) def(x):
    print('sq:', x ** 2)

In [ ]:
x = 2
x
..match:
    case 1:
        print('one')
    case 2:
        print('two')

## Заметки и ограничения v1

- Лямбда разрешена только в трёх statement-позициях: `name = def(...)`, `return def(...)`,
  `def(...)` отдельной инструкцией. Аргументом вызова или элементом списка — нет.
- В `return`/выражении функция получает синтетическое имя `<lambda>`; в присваивании —
  имя цели (`fib.__name__ == 'fib'`).
- `match x = 5` и `match _ as y = 5` — SyntaxError (связывающая цель не может стоять прямо
  перед `=`); писать `match (x) = 5` / `match (_ as y) = 5`.
- `match = 1` и `match.x = 1` остались обычными операциями (мягкое ключевое слово).
- Спеки: `goal-and-requirements.md`, `architecture.md`, `specs/`, план — `IMPLEMENTATION.md`.